# humor-me: decisions and the experiments behind them

**One place for every decision.** The written log is [`docs/agent/decisions-detail.md`](../docs/agent/decisions-detail.md). Its **evidence** is split by
section into notebooks that re-run each experiment, show the table or figure, and **check the claim** the decision depends on.
This notebook is the index: it lists every decision and links each one to the notebook that holds its evidence.

| Notebook | Holds |
|---|---|
| [decisions-data.ipynb](decisions-data.ipynb) | Transcripts and rJokes: Exp 1, 2, 9, 10, 11, 12 |
| [decisions-director.ipynb](decisions-director.ipynb) | Simulator and director: Exp 3 to 8, 13 to 16, 18 |
| [decisions-live.ipynb](decisions-live.ipynb) | The live loop: Exp 19 onward, except judge, SFT and DPO |
| [decisions-model.ipynb](decisions-model.ipynb) | Judge, SFT and DPO: Exp 36 and 37 (the pair judge's gate, Haiku then Opus) |
| [decision-plot.ipynb](decision-plot.ipynb) | The decisions as charts |

Experiment numbers are global: an `Exp N` heading appears in exactly one notebook. Run everything with `python scripts/run_notebook.py`;
one notebook with `python scripts/run_notebook.py notebooks/decisions-data.ipynb`.

Vocabulary: [glossary](../docs/wiki/glossary.md).

In [1]:
import re
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"))
from decisions_common import sections, show

## Index: every decision

Parsed live from `docs/agent/decisions-detail.md`, so it can't go stale. **Evidence** links to the section notebook that re-runs it;
"rationale" means the decision rests on reasoning or constraints, not an experiment (the reasoning is in the log).

In [2]:
EVIDENCE = {  # title fragment in decisions-detail.md -> section heading; its Exp number picks the notebook
    "Exp 24:": "Exp 24 Generator models",
    "Exp 24:": "Exp 24 Generator models",
    "S4: Gemini directly on whole sets": "Exp 1 Bit segmentation",
    "Bit = smallest standalone unit": "Exp 1 Bit segmentation",
    "Pause the transcript pipeline": "Exp 2 rJokes prep",
    "picks a mode per bit": "Exp 3 Per-joke vs per-bit",
    "Director policy: LinUCB": "Exp 4 Policies × priors",
    "Keep the director's features": "Exp 5 Mid-set shift",
    "Keep 8-bit sets": "Exp 6 Set length vs shift",
    "Show state step A": "Exp 7 Show state A",
    "Show state step B": "Exp 8 Show state B",
    "rJokes is for mode statistics only": "Exp 9 R2 labels vs a hand check",
    "Human vs judge mode ranking": "Exp 10 Human vs judge mode ranking",
    "Human `base` for the sim": "Exp 10 Human vs judge mode ranking",
    "rJokes correlation map": "Exp 12 rJokes correlation map",
    "Joke length vs upvotes": "Exp 12 rJokes correlation map",
    "Correlation-map rule misses": "Exp 12 rJokes correlation map",
    "Two tastes in the sim": "Exp 13 Two tastes in the sim",
    "Show state step C1b": "Exp 15 Show state step C1b",
    "Show state step C1": "Exp 14 Show state step C1",
    "Oracle vs planning optimum": "Exp 16 Oracle vs planning optimum",
    "Explore at ε = 0.2": "Exp 18 Cost of exploring at ε",
    "Exp 18:": "Exp 18 Cost of exploring at ε",
    "Label the R2 sample with a room state": "Exp 25 Room labels for the judge's pairs",
    "Safety filter: rules": "Exp 19 Safety filter: rules and an LLM check",
    "Generator: one call per bit": "Exp 20 Generated show timing",
    "Exp 21:": "Exp 21 Click alignment",
    "Exp 22:": "Exp 22 Acceptance run",
    "Exp 23:": "Exp 23 Cached bits fill the show-open silence",
    "Generator: one thread per call": "Exp 26 Per-thread timing",
    "count the jokes in the schema": "Exp 27 Connected threads",
    "skip lines the listener has already heard": "Exp 28 Bridge lines the listener has heard",
    "Log where each bit's wait goes": "Exp 29 Where a bit's wait goes",
    "rewrite it from the spot-check": "Exp 30 Judge v2 on a fresh sample",
    "guess its mode after joke 3": "Exp 31 Next bit: decide at joke 3, or guess it",
    "keep a show ledger": "Exp 32 Show ledger",
    "Give each bit a topic and two robot facts": "Exp 33 Topics and robot facts for each bit",
    "Try the topic alone": "Exp 34 Topics only, no robot facts",
    "Give the writer tonight's context": "Exp 35 Tonight's context for each bit",
    "Public figures in the news may be named": "Exp 35 Tonight's context for each bit",
    "DPO pairs: write them in the background": "Exp 36 Does the judge agree with you on same-writer pairs?",
    "Try Claude Opus 5.5 as the judge": "Exp 37 Does Claude Opus 5.5 agree with you on the same pairs? + Exp 39 Do other people pick like you, or like the judges?",
    "The name check stops rejecting ordinary words": "Exp 40 The name check and ordinary words",
}
where = sections()


def evidence(section):
    if section is None:
        return "rationale"
    links = []
    for one in section.split(" + "):  # a decision can have more than one experiment
        (heading, notebook), = where[int(re.match(r"Exp (\d+)", one).group(1))]
        links.append(f"[{heading}]({notebook})")
    return ", ".join(links)


log = Path("docs/agent/decisions-detail.md").read_text(encoding="utf-8")
entries = re.findall(r'^## <a id="d-\d+"></a>(D-\d+) · (\d{4}-\d{2}-\d{2}) — (.+)$', log, re.M)
# a heading format change once emptied this table without failing anything
assert entries and len(entries) == len(re.findall(r"^## ", log, re.M)), "decisions-detail.md headings no longer parse"
rows = []
for did, date, title in entries:
    ev = evidence(next((v for k, v in EVIDENCE.items() if k in title), None))
    rows.append(f"| {did} | {date} | {title} | {ev} |")
show("| ID | Date | Decision | Evidence |\n|---|---|---|---|\n" + "\n".join(rows))


| ID | Date | Decision | Evidence |
|---|---|---|---|
| D-01 | 2026-10-20 | Audio-only realtime loop | rationale |
| D-02 | 2026-10-20 | Per-joke laughter is the reward; episode rating is the outcome | rationale |
| D-03 | 2026-10-20 | Contextual bandit over HumorGen persona modes, fixed core identity | rationale |
| D-04 | 2026-10-20 | LLM-as-judge first, learned reward model later | rationale |
| D-05 | 2026-10-20 | Data sources | rationale |
| D-06 | 2026-10-20 | Research framing | rationale |
| D-07 | 2026-10-20 | M1 = Dec 15 (RL course final project, school demo) | rationale |
| D-08 | 2026-10-20 | Context = three operator-set features, not one venue label | rationale |
| D-09 | 2026-10-20 | Teens = stricter safety filter (hard rule) | rationale |
| D-10 | 2026-10-20 | Model output = next bit, conditioned on room state | rationale |
| D-11 | 2026-10-20 | Transcripts are for structure + reward, never as imitation text | rationale |
| D-12 | 2026-10-20 | Laugh labels: lexicon-mapped levels, binary primary | rationale |
| D-13 | 2026-10-20 | Opener/closer are positions, not bit types | rationale |
| D-14 | 2026-10-20 | Density gate for laugh labels; Birbiglia replaces Bamford in held-out | rationale |
| D-15 | 2026-10-21 | Applause alone is not a laugh | rationale |
| D-16 | 2026-10-21 | Two datasets, split logically at the output | rationale |
| D-17 | 2026-10-21 | Callbacks are first-class: hooks in set memory, callback(k) planner action | rationale |
| D-18 | 2026-10-21 | Laugh predictor trains on lines, not beats | rationale |
| D-19 | 2026-10-21 | Gemini (free tier) for S4–S5, with minimal excerpts | rationale |
| D-20 | 2026-10-21 | S4: Gemini directly on whole sets (supersedes "minimal excerpts" above) | [Exp 1 Bit segmentation on stand-up transcripts (2026-10-21)](decisions-data.ipynb) |
| D-21 | 2026-10-21 | Bit = smallest standalone unit; same-topic bits form a thread | [Exp 1 Bit segmentation on stand-up transcripts (2026-10-21)](decisions-data.ipynb) |
| D-22 | 2026-10-26 | Pause the transcript pipeline; rJokes + simulator first | [Exp 2 rJokes prep (2026-10-26: pause transcripts, rJokes + simulator first)](decisions-data.ipynb) |
| D-23 | 2026-10-26 | Long-form sets are planned hierarchically, not generated in one pass | rationale |
| D-24 | 2026-10-26 | Northeastern Explorer for GPU work; laptop for everything else | rationale |
| D-25 | 2026-10-27 | The director picks a mode per bit, not per joke | [Exp 3 Per-joke vs per-bit decisions (2026-10-27)](decisions-director.ipynb) |
| D-26 | 2026-10-27 | Director policy: LinUCB with a uniform prior | [Exp 4 Director policy × prior (2026-10-27)](decisions-director.ipynb) |
| D-27 | 2026-10-28 | Keep the director's features; within-show adaptation needs show state | [Exp 5 Mid-set preference shift (2026-10-28)](decisions-director.ipynb) |
| D-28 | 2026-10-28 | Keep 8-bit sets; longer sets don't let the director read a turn | [Exp 6 Does a longer set let the director read a turn? (2026-10-28)](decisions-director.ipynb) |
| D-29 | 2026-10-28 | Show state step A (tonight's record as features): not adopted; go to B | [Exp 7 Show state, step A: does remembering tonight help? (2026-10-28)](decisions-director.ipynb) |
| D-30 | 2026-10-31 | Show state step B (career + tonight's correction): not adopted; C needs exploration and fatigue planning | [Exp 8 Show state, step B: career estimate + tonight's correction (2026-10-31)](decisions-director.ipynb) |
| D-31 | 2026-10-31 | Sim `base`: HumorGen win rates mapped linearly (retroactive record) | rationale |
| D-32 | 2026-10-31 | rJokes is for mode statistics only, never joke material | [Exp 9 R2 labels vs a hand check (2026-10-31)](decisions-data.ipynb) |
| D-33 | 2026-10-31 | Quote a few rJokes rows in docs; never the dataset | rationale |
| D-34 | 2026-11-01 | Human vs judge mode ranking: design and pass criteria (fixed before running) | [Exp 10 Human vs judge mode ranking (2026-11-01)](decisions-data.ipynb) |
| D-35 | 2026-11-01 | Human `base` for the sim: decile win rates | [Exp 10 Human vs judge mode ranking (2026-11-01)](decisions-data.ipynb) |
| D-36 | 2026-11-01 | Human vs judge mode ranking: outcome | [Exp 10 Human vs judge mode ranking (2026-11-01)](decisions-data.ipynb) |
| D-37 | 2026-11-01 | Joke length vs upvotes: weak correlation, clear trend | [Exp 12 rJokes correlation map (2026-11-01)](decisions-data.ipynb) |
| D-38 | 2026-11-01 | rJokes correlation map: design and rule (fixed before running) | [Exp 12 rJokes correlation map (2026-11-01)](decisions-data.ipynb) |
| D-39 | 2026-11-01 | rJokes correlation map: outcome, Exp 10 stands | [Exp 12 rJokes correlation map (2026-11-01)](decisions-data.ipynb) |
| D-40 | 2026-11-01 | Correlation-map rule misses a weak link paired with a strong one | [Exp 12 rJokes correlation map (2026-11-01)](decisions-data.ipynb) |
| D-41 | 2026-11-01 | Two tastes in the sim: pass criteria (fixed before running) | [Exp 13 Two tastes in the sim (2026-11-01)](decisions-director.ipynb) |
| D-42 | 2026-11-01 | Two tastes in the sim: outcome, the director choice holds | [Exp 13 Two tastes in the sim (2026-11-01)](decisions-director.ipynb) |
| D-43 | 2026-11-01 | Demo voice: Kokoro, local, on the CPU | rationale |
| D-44 | 2026-11-01 | M1 target moves to Dec 8; presentation and report on Dec 14 | rationale |
| D-45 | 2026-11-01 | Cut banter and roast handling from M1 | rationale |
| D-46 | 2026-11-01 | Split show state step C: keep the MDP, defer callbacks | rationale |
| D-47 | 2026-11-01 | Show state step C1: method and pass criteria (fixed before running) | [Exp 14 Show state step C1, first run: linear Q-learning (2026-11-01)](decisions-director.ipynb) |
| D-48 | 2026-11-01 | Show state step C1, first run: fails, and the run doesn't test planning fairly | [Exp 14 Show state step C1, first run: linear Q-learning (2026-11-01)](decisions-director.ipynb) |
| D-49 | 2026-11-01 | Show state step C1b: fix the scale and targets (fixed before running) | [Exp 15 Show state step C1b: scale and targets fixed (2026-11-01)](decisions-director.ipynb) |
| D-50 | 2026-11-01 | Show state step C1b, bug fixed: one more run (fixed before running) | [Exp 15 Show state step C1b: scale and targets fixed (2026-11-01)](decisions-director.ipynb) |
| D-51 | 2026-11-01 | rJokes may also test judges, still never joke material | rationale |
| D-52 | 2026-11-01 | Trained joke model and DPO move into M1, from our own premises | rationale |
| D-53 | 2026-11-01 | Oracle vs planning optimum: keep the oracle as the yardstick, report the true ceiling beside it | [Exp 16 Oracle vs planning optimum (2026-11-01)](decisions-director.ipynb) |
| D-54 | 2026-11-01 | Every experiment gets a chart or a stated reason | rationale |
| D-55 | 2026-11-08 | Live loop: one process, with an append-only log as the contract | rationale |
| D-56 | 2026-11-08 | Director state is rebuilt from logs at show start | rationale |
| D-57 | 2026-11-08 | Live reward comes from operator keys, on frozen laugh levels | rationale |
| D-58 | 2026-11-08 | Explore at ε = 0.2 and pick candidates uniformly, with logged probabilities | [Exp 18 Cost of exploring at ε (2026-11-08)](decisions-director.ipynb) |
| D-59 | 2026-11-08 | Record raw show audio, with consent | rationale |
| D-60 | 2026-11-08 | Laugh detector arrives in stages, with buttons as the fallback | rationale |
| D-61 | 2026-11-08 | M1 shows validate the judge and DPO; training waits for about 30 shows | rationale |
| D-62 | 2026-11-08 | PR standard: one task, reviewed before it reaches the user | rationale |
| D-63 | 2026-11-08 | Safety filter: rules first, classifier only if they miss (Exp 19) | [Exp 19 Safety filter: rules and an LLM check (2026-11-08)](decisions-live.ipynb) |
| D-64 | 2026-11-08 | Scraps from the Loft permission, on record | rationale |
| D-65 | 2026-11-08 | One decisions notebook per project section | rationale |
| D-66 | 2026-11-08 | Joke bank gets bit structure, transcript statistics stay undecided | rationale |
| D-67 | 2026-11-08 | Exp 18: exploring at ε = 0.2 misses its bar in one of six rooms | [Exp 18 Cost of exploring at ε (2026-11-08)](decisions-director.ipynb) |
| D-68 | 2026-11-08 | Keep ε = 0.2; the cost bar no longer covers the doubled-judge stress room | rationale |
| D-69 | 2026-11-08 | Safety filter: rules plus an LLM check | [Exp 19 Safety filter: rules and an LLM check (2026-11-08)](decisions-live.ipynb) |
| D-70 | 2026-11-08 | Safety filter: rules plus an LLM check (Exp 19 re-run) | [Exp 19 Safety filter: rules and an LLM check (2026-11-08)](decisions-live.ipynb) |
| D-71 | 2026-11-08 | Safety check latency bar moves to 3 s at p95; red-team list stays in the repo | rationale |
| D-72 | 2026-11-08 | Safety bar: at least 75% on fresh indirect lines; no pre-approval of scripts | rationale |
| D-73 | 2026-11-08 | Safety bar met on the second fresh set; T8 accepted | rationale |
| D-74 | 2026-11-14 | Generator: one call per bit, render-ahead, bridge lines and a bank fallback (Exp 20) | [Exp 20 Generated show timing (2026-11-14)](decisions-live.ipynb) |
| D-75 | 2026-11-14 | Bank fallback lines run through the rules only (user) | rationale |
| D-76 | 2026-11-14 | A bit plays one candidate thread (user) | rationale |
| D-77 | 2026-11-14 | Session log v2 adds a bridge event (user) | rationale |
| D-78 | 2026-11-14 | Exp 20: the generated show meets its timing bars on three shows | rationale |
| D-79 | 2026-11-15 | Consent-gated raw audio over the websocket (T10) | rationale |
| D-80 | 2026-11-15 | Exp 21: the click-alignment bar re-tuned to the spread around a constant (user) | [Exp 21 Click alignment (2026-11-15)](decisions-live.ipynb) |
| D-81 | 2026-11-15 | Operator UI waveform: full-width bars with a solid idle line, lilac-tinted mono theme (user) | rationale |
| D-82 | 2026-11-15 | First live show uses Gemini (GENERATOR=llm) (user) | rationale |
| D-83 | 2026-11-15 | Exp 22: rating after show_end; the acceptance scorer is built, the shows are the user's | [Exp 22 Acceptance run (2026-11-15)](decisions-live.ipynb) |
| D-84 | 2026-11-15 | Exp 22: review round 1 changes to the rating and the scorer | [Exp 22 Acceptance run (2026-11-15)](decisions-live.ipynb) |
| D-85 | 2026-11-15 | Exp 22: the acceptance run passes; Stage 0 is done | [Exp 22 Acceptance run (2026-11-15)](decisions-live.ipynb) |
| D-86 | 2026-11-24 | Split the decision log: plain log for humans, detail log for agents (user) | rationale |
| D-87 | 2026-11-24 | Docs people read are short and plain; agent detail moves to docs/agent/ (user) | rationale |
| D-88 | 2026-11-15 | The safety check gets a budget sized to a stall, and re-asks once on a bad reply | rationale |
| D-89 | 2026-11-24 | Label the R2 sample with a room state for Tingyu's DPO judge (user) | [Exp 25 Room labels for the judge's pairs (2026-11-24)](decisions-data.ipynb) |
| D-90 | 2026-11-24 | A paused or cut-short show keeps its usable parts; pauses get log rows (user) | rationale |
| D-91 | 2026-11-24 | Exp 23: cached popular bits fill the show-open and between-bit silences (user) | [Exp 23 Cached bits fill the show-open silence (2026-11-24)](decisions-live.ipynb) |
| D-92 | 2026-11-24 | A joke whose playback overlapped a pause loses its label from the reward (user) | rationale |
| D-93 | 2026-11-24 | Model settings are named by job, and the joke writer can run on Claude through the official SDK (user) | rationale |
| D-94 | 2026-11-24 | Exp 24: compare Flash-Lite, Flash and Claude Haiku 5.5 as the bit generator (user) | [Exp 24 Generator models (2026-11-24; candidates changed 2026-11-25, before any scoring)](decisions-live.ipynb) |
| D-95 | 2026-11-24 | Exp 24: a judge calibrated on the user's votes may rank prompts 31 to 60 (user) | [Exp 24 Generator models (2026-11-24; candidates changed 2026-11-25, before any scoring)](decisions-live.ipynb) |
| D-96 | 2026-11-25 | Exp 24: replace the Gemini writers with two local Ollama models, keep Claude Haiku 5.5 (user) | [Exp 24 Generator models (2026-11-24; candidates changed 2026-11-25, before any scoring)](decisions-live.ipynb) |
| D-97 | 2026-11-25 | Exp 24: a local candidate must fit beside Kokoro; gemma4:12b joins as a reference (user) | [Exp 24 Generator models (2026-11-24; candidates changed 2026-11-25, before any scoring)](decisions-live.ipynb) |
| D-98 | 2026-11-25 | Exp 19 rerun: Claude Haiku 5.5 as the safety filter keeps the catch rate but misses the 3 s bar (user) | rationale |
| D-99 | 2026-11-25 | Exp 19 rerun: Claude Haiku with thinking off still misses the 3 s bar (user) | rationale |
| D-100 | 2026-11-25 | Exp 19 rerun: the safety filter moves to Claude Haiku with thinking off, under a 7 s bar (user) | rationale |
| D-101 | 2026-11-25 | Exp 24: send local models the reply schema, and replace mimo-v2.6:9b with qwen3:4b-instruct (user) | [Exp 24 Generator models (2026-11-24; candidates changed 2026-11-25, before any scoring)](decisions-live.ipynb) |
| D-102 | 2026-11-25 | DPO's judge is the D-95 judge, taught on 60 same-model pairs and gated on 60 more (user) | rationale |
| D-103 | 2026-11-25 | Exp 24: bonsai's first block is archived, not scored, after a local server fault; its block is rerun (user) | [Exp 24 Generator models (2026-11-24; candidates changed 2026-11-25, before any scoring)](decisions-live.ipynb) |
| D-104 | 2026-11-25 | Exp 24: drop the gemma4:12b reference, which cannot finish a bit in 12 s (user) | [Exp 24 Generator models (2026-11-24; candidates changed 2026-11-25, before any scoring)](decisions-live.ipynb) |
| D-105 | 2026-11-25 | Safety rules read curly punctuation as straight; Exp 24's safety step is redone and the started vote is voided (user) | rationale |
| D-106 | 2026-11-25 | Exp 24's ballot and judge show punctuation in plain form, so no model is given away (user) | rationale |
| D-107 | 2026-11-25 | Exp 24: bonsai-2:4b writes the live show's bits and is the SFT/DPO base; Haiku is the funnier reference to beat (user) | [Exp 24 Generator models (2026-11-24; candidates changed 2026-11-25, before any scoring)](decisions-live.ipynb) |
| D-108 | 2026-11-25 | Exp 24: an exact tie in mean rank goes to the faster model, not the first name (user) | [Exp 24 Generator models (2026-11-24; candidates changed 2026-11-25, before any scoring)](decisions-live.ipynb) |
| D-109 | 2026-11-25 | bonsai-2:4b writes the show through Ollama; each show loads it first and refuses to start unless it is all on the GPU (user) | rationale |
| D-110 | 2026-11-25 | Generator: one thread per call, three calls in parallel on one prompt (Exp 26 checks the timing) (user) | [Exp 26 Per-thread timing (2026-11-25)](decisions-live.ipynb) |
| D-111 | 2026-11-25 | Generator: count the jokes in the schema, re-ask a bad shape, and test a connected prompt (Exp 27 decides it) | [Exp 27 Connected threads (2026-11-25)](decisions-live.ipynb) |
| D-112 | 2026-11-25 | Bridge lines: skip lines the listener has already heard, fall back to filler (Exp 28 measures it) (user) | [Exp 28 Bridge lines the listener has heard (2026-11-25)](decisions-live.ipynb) |
| D-113 | 2026-11-25 | Log where each bit's wait goes: a `prep` row per bit and the bridge overhang on its first joke (Exp 29 decides which stage to fix) | [Exp 29 Where a bit's wait goes (2026-11-25)](decisions-live.ipynb) |
| D-114 | 2026-11-25 | Thread judge: rewrite it from the spot-check, keep whichever of the two agrees with the user more (Exp 30 decides it) (user) | [Exp 30 Judge v2 on a fresh sample (2026-11-25)](decisions-live.ipynb) |
| D-115 | 2026-11-26 | Next bit: guess its mode after joke 3 and write it ahead, decide at the bit's end as before (Exp 31) (user) | [Exp 31 Next bit: decide at joke 3, or guess it (2026-11-26)](decisions-director.ipynb) |
| D-116 | 2026-11-26 | Keep a show ledger: one row per logged show, with the user's rating as the outcome, and commit the played lines (Exp 32) (user) | rationale |
| D-117 | 2026-11-26 | Give each bit a topic and two robot facts so the writer stops landing on the robot's hardware (Exp 33) (user) | [Exp 33 Topics and robot facts for each bit (2026-11-26)](decisions-live.ipynb) |
| D-118 | 2026-11-26 | Try the topic alone, with no robot facts, against Exp 33's old bits (Exp 34) (user) | [Exp 34 Topics only, no robot facts (2026-11-26)](decisions-live.ipynb) |
| D-119 | 2026-11-26 | Give the writer tonight's context: approved headline gists, the market line and one optional typed line (Exp 35) (user) | [Exp 35 Tonight's context for each bit (2026-11-26)](decisions-live.ipynb) |
| D-120 | 2026-11-26 | Public figures in the news may be named in tonight's gists and jokes (user) | [Exp 35 Tonight's context for each bit (2026-11-26)](decisions-live.ipynb) |
| D-121 | 2026-11-26 | DPO pairs: write them in the background with the live writer, rank 120 of them yourself, and gate the judge on 60 (Exp 36) (user) | [Exp 36 Does the judge agree with you on same-writer pairs?](decisions-model.ipynb) |
| D-122 | 2026-11-27 | Try Claude Opus 5.5 as the judge on the same 120 pairs, with a stricter bar for the second look (Exp 37) (user) | [Exp 37 Does Claude Opus 5.5 agree with you on the same pairs?](decisions-model.ipynb), [Exp 39 Do other people pick like you, or like the judges?](decisions-model.ipynb) |
| D-123 | 2026-11-27 | Reserved for open PR #90 (DPO training) | rationale |
| D-124 | 2026-11-27 | Reserved for open PR #90 (which model gets trained) | rationale |
| D-125 | 2026-11-27 | The name check stops rejecting ordinary words as people's names (Exp 40) (user) | [Exp 40 The name check and ordinary words (2026-11-27)](decisions-live.ipynb) |

## Adding a new decision

1. Put the experiment in a module as `run(**params) -> dict` and `table(results) -> str`
   (plus `plot` if it has a figure), and call it via `run_or_load(name, run, **params)`.
2. Add a section to the notebook for its area, with the next free global Exp number: simulator and director in
   `decisions-director.ipynb`, transcripts and rJokes in `decisions-data.ipynb`, the live loop in `decisions-live.ipynb`
   (judge, SFT and DPO go in `decisions-model.ipynb`, created with its first experiment). The section is
   **Decision · Where it fits · Question/options · How to read**, a cell that runs it, and **checks** for every claim the
   decision makes. Checks assert the **decision**, including rejections ("option X misses the pre-set criteria → not
   adopted"), so all ✅ always means "every decision still holds". Show per-option pass/fail as a table.
   A new section notebook starts from the setup cell of an existing one and imports only the modules it uses.
3. Add the title fragment to `EVIDENCE` in the index cell here (its value starts with `Exp N`), the entry (next D-xx ID) to
   `docs/agent/decisions-detail.md` with its row in `docs/decisions.md`, and the experiment to `COVERAGE` in `decision-plot.ipynb`.
4. `python scripts/run_notebook.py`, then commit the notebooks with their outputs.